# 1.3 Functional API

這份 Notebook 示範 Functional API 的多輸入模型。範例任務是簡化的房價預測：一組輸入是數值特徵，另一組輸入是地區 one-hot 特徵。


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score

tf.keras.utils.set_random_seed(42)


## 1. 建立多輸入資料

真實專案常同時有數值特徵與類別特徵。這裡用合成資料模擬房價預測。


In [ ]:
rng = np.random.default_rng(42)
n = 1200
size = rng.normal(35, 8, n).clip(15, 70)
rooms = rng.integers(1, 6, n)
age = rng.normal(12, 6, n).clip(0, 35)
area = rng.choice(['north', 'center', 'south'], size=n, p=[0.3, 0.45, 0.25])
area_bonus = {'north': 12, 'center': 25, 'south': 5}
noise = rng.normal(0, 4, n)
price = 1.8 * size + 4.5 * rooms - 0.7 * age + np.vectorize(area_bonus.get)(area) + noise

df = pd.DataFrame({'size': size, 'rooms': rooms, 'age': age, 'area': area, 'price': price})
df.head()


## 2. 切分數值輸入與類別輸入


In [ ]:
numeric_cols = ['size', 'rooms', 'age']
category_col = ['area']

df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

scaler = StandardScaler()
x_train_num = scaler.fit_transform(df_train[numeric_cols]).astype('float32')
x_test_num = scaler.transform(df_test[numeric_cols]).astype('float32')

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
x_train_area = encoder.fit_transform(df_train[category_col]).astype('float32')
x_test_area = encoder.transform(df_test[category_col]).astype('float32')

y_train = df_train['price'].to_numpy(dtype='float32')
y_test = df_test['price'].to_numpy(dtype='float32')

print('numeric input:', x_train_num.shape)
print('area input:', x_train_area.shape)


## 3. 使用 Functional API 建立多輸入模型


In [ ]:
numeric_input = tf.keras.Input(shape=(x_train_num.shape[1],), name='numeric_features')
area_input = tf.keras.Input(shape=(x_train_area.shape[1],), name='area_features')

x = tf.keras.layers.Dense(32, activation='relu')(numeric_input)
x = tf.keras.layers.Dense(16, activation='relu')(x)
combined = tf.keras.layers.Concatenate()([x, area_input])
combined = tf.keras.layers.Dense(16, activation='relu')(combined)
output = tf.keras.layers.Dense(1, name='price')(combined)

model = tf.keras.Model(inputs=[numeric_input, area_input], outputs=output)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss='mse', metrics=['mae'])
model.summary()


## 4. 訓練模型


In [ ]:
history = model.fit(
    {'numeric_features': x_train_num, 'area_features': x_train_area},
    y_train,
    validation_split=0.2,
    epochs=80,
    batch_size=32,
    verbose=0
)


## 5. 評估模型


In [ ]:
train_result = model.evaluate({'numeric_features': x_train_num, 'area_features': x_train_area}, y_train, verbose=0, return_dict=True)
test_result = model.evaluate({'numeric_features': x_test_num, 'area_features': x_test_area}, y_test, verbose=0, return_dict=True)

pd.DataFrame([train_result, test_result], index=['train', 'test'])


In [ ]:
pred = model.predict({'numeric_features': x_test_num, 'area_features': x_test_area}, verbose=0).ravel()
print('MAE:', mean_absolute_error(y_test, pred))
print('R2:', r2_score(y_test, pred))

pd.DataFrame({'actual': y_test[:8], 'prediction': pred[:8]})


## 6. 小結

Functional API 讓我們能處理多輸入、多輸出與分支結構。只要任務不再是單一路徑模型，就應該優先考慮 Functional API。
